# Display an ENVI classification map

A small helper to view the classification images produced by `02_batch_predict_image.ipynb`, with a color-coded, labeled legend. Use it to inspect results or to grab a figure for a report.

**Display function.** Opens an ENVI classification `.hdr`, reads the class names from its metadata, assigns each class a distinct color (Empty and the unidentified `Genus_species` placeholder get greys), and plots the map with a legend.

In [ ]:
import spectral
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches


def display_envi_classification(hdr_path):
    """
    Open and display an ENVI classification image with a labeled legend.

    Args:
        hdr_path (str): Path to the .hdr file of the classification image
            (as written by 02_batch_predict_image.ipynb).
    """
    # --- 1. Open the ENVI classification file ---
    try:
        img = spectral.open_image(hdr_path)
        print(f"Successfully opened: {hdr_path}")
        print(f"Image dimensions (Rows, Cols, Bands): {img.shape}")

        # --- 2. Load the class map as a 2D array ---
        class_map = img.load()
        if class_map.ndim == 3 and class_map.shape[2] == 1:
            class_map = class_map[:, :, 0]
    except Exception as e:
        print(f"Error: could not open or read '{hdr_path}'.")
        print(f"Details: {e}")
        return

    # --- 3. Class names live in the header metadata ---
    class_names = img.metadata.get('class names', [])
    if not class_names:
        print("Warning: no 'class names' in the header; using generic names.")
        num_classes = int(class_map.max()) + 1
        class_names = [f'Class {i}' for i in range(num_classes)]

    # --- 4. Build a categorical colormap ---
    # Bright, well-separated colors (Sasha Trubetskoy's "20 distinct colors").
    # Greys/white are omitted so they do not clash with the greys used for the
    # Empty value and the unidentified 'Genus_species' placeholder below.
    vivid_palette = [
        '#e6194B', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4',
        '#f032e6', '#bfef45', '#fabed4', '#469990', '#9A6324', '#ffe119',
        '#000075', '#808000', '#aaffc3', '#ffd8b1',
    ]
    if len(class_names) > len(vivid_palette):
        extra = plt.cm.get_cmap('gist_ncar', len(class_names))
        cmap_colors = [extra(i) for i in range(len(class_names))]
    else:
        cmap_colors = list(vivid_palette[:len(class_names)])

    if 'Genus_species' in class_names:
        cmap_colors[class_names.index('Genus_species')] = '#C8C8C8'

    # Prepend grey for the Empty value (-1).
    colors = ['#808080'] + list(cmap_colors)
    custom_cmap = ListedColormap(colors)
    bounds = np.arange(len(colors) + 1) - 1.5
    norm = plt.matplotlib.colors.BoundaryNorm(bounds, len(colors))

    # --- 5. Plot with a legend ---
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(class_map, cmap=custom_cmap, norm=norm)
    ax.set_title(f"Classification Map: {img.metadata.get('description', 'Untitled')}", fontsize=16)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

    legend_labels = ['Empty'] + class_names
    patches = [mpatches.Patch(color=colors[i], label=legend_labels[i])
               for i in range(len(legend_labels))]
    ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()


**Pick a map and display it.** Reads `config.yaml` to rebuild the path to a classification map for the chosen `TASK`. Change `TASK`, or set `file_to_display` directly to view any ENVI classification image.

In [ ]:
import os
import yaml
from pathlib import Path

# config.yaml (and the paths inside it) are relative to the repo root, but
# this notebook lives in notebooks/. Locate the repo root and resolve every
# configured path against it so it works from either directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'config.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

with open(REPO_ROOT / 'config.yaml') as f:
    CONFIG = yaml.safe_load(f)
for _section in ('paths', 'prediction'):
    for _key, _val in CONFIG.get(_section, {}).items():
        if isinstance(_val, str):
            CONFIG[_section][_key] = str(REPO_ROOT / _val)

# Which task's classification map to display: plant, age, part, health, lifecycle.
TASK = 'plant'

# The prediction notebook names outputs "<image>_<task>_classification.hdr" in
# the configured output directory. Rebuild that path here.
base = os.path.splitext(os.path.basename(CONFIG['prediction']['input_hdr']))[0]
file_to_display = os.path.join(CONFIG['prediction']['output_dir'],
                               f"{base}_{TASK}_classification.hdr")

# Or set file_to_display to any ENVI classification .hdr you want to view.
display_envi_classification(file_to_display)
